1. IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

2. LOAD DATA

In [2]:
df = pd.read_csv("../ml/data/household_indonesia_dataset.csv", parse_dates=["datetime"])
df.sort_values("datetime", inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

Loaded 50,000 rows × 57 columns


3. FEATURE ENGINEERING

In [3]:
# Encode golongan_tarif → ordinal
df["golongan_enc"] = df["golongan_tarif"].map({"R-1/TR": 0, "R-2/TR": 1, "R-3/TR": 2})

In [4]:
#  Rasio beban
df["rasio_ac_luas"]        = df["jumlah_ac"] * 900 / (df["luas_rumah"] + 1)
df["rasio_penghuni_luas"]  = df["jumlah_penghuni"] / (df["luas_rumah"] + 1)
df["watt_total_terpasang"] = (
    df["jumlah_ac"]         * 900 +
    df["jumlah_kulkas"]     * 100 +
    df["jumlah_tv"]         * 70  +
    df["jumlah_lampu"]      * 10  +
    df["jumlah_mesin_cuci"] * 500 +
    df["jumlah_komputer"]   * 120 +
    df["water_heater"]      * 2000 +
    df["dispenser"]         * 350  +
    df["microwave"]         * 1000
)

In [5]:
# Histori rata-rata & std (dari kolom hist_h1..hist_h7)
hist_cols = [f"hist_h{i}" for i in range(1, 8)]
df["hist_mean"] = df[hist_cols].mean(axis=1).round(3)
df["hist_std"]  = df[hist_cols].std(axis=1).fillna(0).round(3)
df["hist_trend"]= (df["hist_h1"] - df["hist_h7"]).round(3)  # tren 7 hari

# Faktor suhu terhadap AC
df["suhu_x_ac_jam"] = df["suhu"] * df["jumlah_ac"] * df["jam_ac_per_hari"]

# Lag histori (H-1)
df["lag_kwh_1hari"] = df["hist_h1"]

print(f"Shape setelah FE: {df.shape}")

Shape setelah FE: (50000, 66)


4. DEFINE FEATURES & TARGET

In [6]:
FEATURES = [
    # ── Waktu ──────────────────────────────────────────────────────
    "jam", "minute", "day_of_week", "day_of_month", "month",
    "quarter", "week_of_year", "is_weekend", "hari_libur",
    "time_of_day", "season",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    # ── Lingkungan & rumah ─────────────────────────────────────────
    "suhu", "jumlah_penghuni", "luas_rumah", "daya_listrik", "golongan_enc",
    # ── Perangkat ──────────────────────────────────────────────────
    "jumlah_ac",     "jam_ac_per_hari",
    "jumlah_kulkas",
    "jumlah_tv",     "jam_tv_per_hari",
    "jumlah_lampu",  "jam_lampu_per_hari",
    "jumlah_mesin_cuci", "frekuensi_cuci_per_minggu",
    "jumlah_komputer",   "jam_komputer_per_hari",
    "jumlah_perangkat_aktif",
    "water_heater", "dispenser", "microwave",
    # ── kWh per perangkat ──────────────────────────────────────────
    "kwh_ac", "kwh_kulkas", "kwh_tv", "kwh_lampu",
    "kwh_mesin_cuci", "kwh_komputer", "kwh_tambahan",
    # ── Fitur turunan ──────────────────────────────────────────────
    "rasio_ac_luas", "rasio_penghuni_luas", "watt_total_terpasang",
    "suhu_x_ac_jam",
    # ── Histori ────────────────────────────────────────────────────
    *[f"hist_h{i}" for i in range(1, 8)],
    "hist_mean", "hist_std", "hist_trend", "lag_kwh_1hari",
]

TARGET_DAY = "konsumsi_kwh_hari"
TARGET_JAM = "konsumsi_kw_jam"

print(f"Jumlah fitur: {len(FEATURES)}")

Jumlah fitur: 59


5. SPLIT 70/15/15

In [7]:
print("\n" + "=" * 65)
print("STEP 3 │ Chronological Split 70/15/15")
print("=" * 65)

n  = len(df)
t1 = int(n * 0.70)
t2 = int(n * 0.85)

X         = df[FEATURES]
y_day     = df[TARGET_DAY]
y_jam     = df[TARGET_JAM]

X_train, y_day_train, y_jam_train = X.iloc[:t1], y_day.iloc[:t1], y_jam.iloc[:t1]
X_val,   y_day_val,   y_jam_val   = X.iloc[t1:t2], y_day.iloc[t1:t2], y_jam.iloc[t1:t2]
X_test,  y_day_test,  y_jam_test  = X.iloc[t2:], y_day.iloc[t2:], y_jam.iloc[t2:]

print(f"Train : {len(X_train):>8,}  Val : {len(X_val):>7,}  Test : {len(X_test):>7,}")

scaler = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_val   = scaler.transform(X_val)
Xs_test  = scaler.transform(X_test)


STEP 3 │ Chronological Split 70/15/15
Train :   35,000  Val :   7,500  Test :   7,500


5. TRAIN — MODEL 1: kWh/hari

In [8]:
xgb_params = dict(
    n_estimators=600, learning_rate=0.05, max_depth=8,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=1.0,
    objective="reg:squarederror", eval_metric="rmse",
    early_stopping_rounds=40, n_jobs=-1, random_state=42, verbosity=0,
)

model_day = xgb.XGBRegressor(**xgb_params)
model_day.fit(Xs_train, y_day_train,
              eval_set=[(Xs_val, y_day_val)], verbose=100)

[0]	validation_0-rmse:17.34322
[100]	validation_0-rmse:0.52204
[200]	validation_0-rmse:0.51037
[202]	validation_0-rmse:0.51034


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.85
,early_stopping_rounds,40
,enable_categorical,False
,eval_metric,'rmse'
,feature_types,None


6. TRAIN — MODEL 2: kW saat ini

In [9]:
model_jam = xgb.XGBRegressor(**xgb_params)
model_jam.fit(Xs_train, y_jam_train,
              eval_set=[(Xs_val, y_jam_val)], verbose=100)

[0]	validation_0-rmse:0.41862
[100]	validation_0-rmse:0.03216
[200]	validation_0-rmse:0.03165
[245]	validation_0-rmse:0.03165


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.85
,early_stopping_rounds,40
,enable_categorical,False
,eval_metric,'rmse'
,feature_types,None


7. EVALUASI

In [10]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-9))) * 100
    print(f"  [{name}]  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%")
    return {"mae": round(mae,5), "rmse": round(rmse,5), "r2": round(r2,5), "mape": round(mape,3)}

print("\n▸ Model kWh/hari:")
m1_val  = evaluate("Val",  y_day_val,  model_day.predict(Xs_val))
m1_test = evaluate("Test", y_day_test, model_day.predict(Xs_test))

print("\n▸ Model kW/jam:")
m2_val  = evaluate("Val",  y_jam_val,  model_jam.predict(Xs_val))
m2_test = evaluate("Test", y_jam_test, model_jam.predict(Xs_test))

# Feature importance top 20
fi = pd.Series(model_day.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\n▸ Top 15 Feature Importance (kWh/hari model):")
print(fi.head(15).to_string())


▸ Model kWh/hari:
  [Val]  MAE=0.3676  RMSE=0.5099  R²=0.9962  MAPE=2.24%
  [Test]  MAE=0.3763  RMSE=0.5238  R²=0.9960  MAPE=2.25%

▸ Model kW/jam:
  [Val]  MAE=0.0202  RMSE=0.0316  R²=0.9946  MAPE=3.59%
  [Test]  MAE=0.0202  RMSE=0.0312  R²=0.9948  MAPE=3.60%

▸ Top 15 Feature Importance (kWh/hari model):
hist_mean        0.752383
hist_h2          0.106763
hist_h1          0.025258
hist_h5          0.022253
hist_h3          0.021669
hist_h4          0.016404
hist_h7          0.014572
hist_h6          0.012762
kwh_ac           0.010068
lag_kwh_1hari    0.009474
suhu_x_ac_jam    0.000631
kwh_kulkas       0.000397
dispenser        0.000374
jumlah_kulkas    0.000339
kwh_tambahan     0.000300


7b. MATH ALGORITHM XGBOOST

In [11]:
print("  Loss     : L(y, ŷ) = (1/2) × Σ(y_i - F(x_i))²")
print("  Gradient : g_i = ∂L/∂F = F(x_i) - y_i")
print("  Hessian  : h_i = ∂²L/∂F² = 1  (konstan untuk MSE)")

F0 = float(y_day_train.mean())
print(f"\n  Prediksi awal  : F₀ = mean(y_train) = {F0:.4f} kWh/hari")

  Loss     : L(y, ŷ) = (1/2) × Σ(y_i - F(x_i))²
  Gradient : g_i = ∂L/∂F = F(x_i) - y_i
  Hessian  : h_i = ∂²L/∂F² = 1  (konstan untuk MSE)

  Prediksi awal  : F₀ = mean(y_train) = 16.7190 kWh/hari


In [12]:
y_sample = y_day_train.iloc[:4].values
print(f"  {'No':<4} {'y_actual':>10} {'F₀':>10} {'residual':>10} {'g_i=F₀-y':>12} {'h_i':>6}")
print("  " + "-" * 55)
g_samples = []
for i, yi in enumerate(y_sample):
    residual = yi - F0
    gi = F0 - yi
    g_samples.append(gi)
    print(f"  {i+1:<4} {yi:>10.4f} {F0:>10.4f} {residual:>10.4f} {gi:>12.4f} {'1.0':>6}")

  No     y_actual         F₀   residual     g_i=F₀-y    h_i
  -------------------------------------------------------
  1       20.9280    16.7190     4.2090      -4.2090    1.0
  2       14.1130    16.7190    -2.6060       2.6060    1.0
  3        8.9740    16.7190    -7.7450       7.7450    1.0
  4        8.6540    16.7190    -8.0650       8.0650    1.0


In [13]:
lam   = model_day.reg_lambda
G_j   = sum(g_samples)
H_j   = float(len(g_samples))
w_star = -G_j / (H_j + lam)
print(f"  λ (reg_lambda) = {lam}")
print(f"  G_j = Σg_i     = {G_j:.4f}")
print(f"  H_j = Σh_i     = {H_j:.1f}")
print(f"  w*  = -{G_j:.4f} / ({H_j:.1f} + {lam}) = {w_star:.4f}")


  λ (reg_lambda) = 1.0
  G_j = Σg_i     = 14.2070
  H_j = Σh_i     = 4.0
  w*  = -14.2070 / (4.0 + 1.0) = -2.8414


In [14]:
G_L, H_L = g_samples[0] + g_samples[1], 2.0
G_R, H_R = g_samples[2] + g_samples[3], 2.0
score_L   = (G_L**2) / (H_L + lam)
score_R   = (G_R**2) / (H_R + lam)
score_T   = ((G_L+G_R)**2) / (H_L + H_R + lam)
gain      = 0.5 * (score_L + score_R - score_T)
print(f"\n  Simulasi (S1,S2 → kiri | S3,S4 → kanan):")
print(f"  G_L={G_L:.4f}, H_L={H_L} | G_R={G_R:.4f}, H_R={H_R}")
print(f"  Gain = 0.5 × [{score_L:.4f} + {score_R:.4f} - {score_T:.4f}]")
print(f"       = {gain:.4f}")



  Simulasi (S1,S2 → kiri | S3,S4 → kanan):
  G_L=-1.6030, H_L=2.0 | G_R=15.8100, H_R=2.0
  Gain = 0.5 × [0.8565 + 83.3188 - 40.3679]
       = 21.9037


In [15]:
eta = model_day.learning_rate
booster  = model_day.get_booster()
trees_df = booster.trees_to_dataframe()
print(f"  η (learning_rate) = {eta}")
print(f"  Jumlah pohon (T)  = {model_day.best_iteration + 1}")
print(f"\n  {'Iterasi t':<12} {'leaf_mean h_t':>15} {'F_t (kumulatif)':>18}")
print("  " + "-" * 47)
F_cur = F0
for t in range(3):
    lv = trees_df[(trees_df['Tree']==t) & (trees_df['Feature']=='Leaf')]['Gain'].mean()
    F_cur += eta * lv
    print(f"  t={t+1:<10} {lv:>15.6f} {F_cur:>18.4f}")
print(f"  ... berlanjut hingga t = {model_day.best_iteration + 1}")


  η (learning_rate) = 0.05
  Jumlah pohon (T)  = 164

  Iterasi t      leaf_mean h_t    F_t (kumulatif)
  -----------------------------------------------
  t=1                 0.853302            16.7617
  t=2                 0.784837            16.8009
  t=3                 0.733539            16.8376
  ... berlanjut hingga t = 164


In [16]:
tree1  = trees_df[trees_df['Tree']==0]
leaves = tree1[tree1['Feature']=='Leaf']
root   = tree1[tree1['Node']==0].iloc[0]
fi_idx = int(str(root['Feature']).replace('f',''))
fname  = FEATURES[fi_idx] if fi_idx < len(FEATURES) else root['Feature']
print(f"  Total node     : {len(tree1)}")
print(f"  Total daun     : {len(leaves)}")
print(f"  Root split on  : {fname}")
print(f"  Root gain      : {root['Gain']:.4f}")
print(f"  Root cover     : {root['Cover']:.0f} sampel")
print(f"  Leaf val range : [{leaves['Gain'].min():.4f}, {leaves['Gain'].max():.4f}]")


  Total node     : 61
  Total daun     : 31
  Root split on  : hist_mean
  Root gain      : 1144478.0000
  Root cover     : 28036 sampel
  Leaf val range : [0.1638, 2.3644]


In [17]:
for k, v in {
    "n_estimators (T)"  : model_day.best_iteration + 1,
    "learning_rate (η)" : model_day.learning_rate,
    "max_depth"         : model_day.max_depth,
    "min_child_weight"  : model_day.min_child_weight,
    "subsample"         : model_day.subsample,
    "colsample_bytree"  : model_day.colsample_bytree,
    "reg_lambda (λ)"    : model_day.reg_lambda,
    "reg_alpha (α)"     : model_day.reg_alpha,
}.items():
    print(f"  {k:<25} : {v}")

  n_estimators (T)          : 164
  learning_rate (η)         : 0.05
  max_depth                 : 8
  min_child_weight          : 5
  subsample                 : 0.8
  colsample_bytree          : 0.85
  reg_lambda (λ)            : 1.0
  reg_alpha (α)             : 0.1


8. THRESHOLDS EFISIENSI

In [18]:
TARIF_PLN = {"R-1/TR": 1444.70, "R-2/TR": 1444.70, "R-3/TR": 1699.53}

q_day = {
    "sangat_efisien": float(y_day.quantile(0.20)),
    "efisien":        float(y_day.quantile(0.40)),
    "sedang":         float(y_day.quantile(0.60)),
    "tinggi":         float(y_day.quantile(0.80)),
}
print("kWh/hari thresholds:", {k: round(v,2) for k,v in q_day.items()})

kWh/hari thresholds: {'sangat_efisien': 9.52, 'efisien': 13.92, 'sedang': 17.68, 'tinggi': 22.08}


9. SAVE MODEL

In [ ]:
joblib.dump(model_day, "model_kwh_hari.joblib")
joblib.dump(model_jam, "model_kw_jam.joblib")
joblib.dump(scaler,    "scaler_indonesia.joblib")

meta = {
    "features": FEATURES,
    "targets":  {"kwh_hari": TARGET_DAY, "kw_jam": TARGET_JAM},
    "thresholds_kwh_hari": q_day,
    "tarif_pln": TARIF_PLN,
    "metrics": {
        "kwh_hari": {"val": m1_val, "test": m1_test},
        "kw_jam":   {"val": m2_val, "test": m2_test},
    },
    "best_iterations": {
        "kwh_hari": int(model_day.best_iteration),
        "kw_jam":   int(model_jam.best_iteration),
    },
    "feature_importance_kwh": fi.head(20).to_dict(),
    "data_stats": {
        "kwh_hari_mean": round(float(y_day.mean()), 3),
        "kwh_hari_std":  round(float(y_day.std()),  3),
        "kwh_hari_min":  round(float(y_day.min()),  3),
        "kwh_hari_max":  round(float(y_day.max()),  3),
        "kw_jam_mean":   round(float(y_jam.mean()), 4),
    }
}

with open(f"model_metadata_indonesia.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"✅ model_kwh_hari.joblib")
print(f"✅ model_kw_jam.joblib")
print(f"✅ scaler_indonesia.joblib")
print(f"✅ model_metadata_indonesia.json")

FileNotFoundError: [Errno 2] No such file or directory: '../saved/model_kwh_hari.joblib'